<!-- nav -->
[← 8.11 · POSIX BRE](08_11_POSIX_BRE.ipynb) · [Home](../../README.md) · [8.13 · SystemVerilog Types →](08_13_SystemVerilog_Types.ipynb)

# 8.12 · POSIX ERE

POSIX's **extended regular expressions** (IEEE Std 1003.1, XBD 9.4) write their operators as themselves, `(a|b)+c?`,
where a BRE escapes them ([8.11 · POSIX BRE](08_11_POSIX_BRE.ipynb)): `|` alternation, `+` and `?` repetitions, `{1,3}`
intervals; they stand an anchor anywhere, and have no backreferences. `egrep` and `grep -E` read them, as a list of
patterns a line each, and `awk` reads them in its programs, writing their characters with C's escapes. `POSIX ERE` is
the dialect, of the POSIX family beside `POSIX BRE`, each reading the other; and C++'s `std::regex` reads its terms when
its flags name the `extended`, `awk` or `egrep` grammar ([8.10 · Cpp RegExp](08_10_Cpp_RegExp.ipynb), 08.10.4).

This page builds an ERE, then what its grammar has that a BRE's has not, and what it lacks; then egrep's lists and awk's
escapes, what validation says, how C++ reads an ERE, and the editions it covers. It starts from `dialects()`; `S` is its
holder namespace, and `char`, `alternative`, `pattern` and `anchor` are the specs of the previous pages:

In [1]:
import assert from "node:assert/strict";
import * as CppRegExp from "@mbse/expressions/Cpp/RegExp";
import { Simplifiers, Standards, Symbolics, Terms } from "@mbse/expressions/Framework";
import * as BRE from "@mbse/expressions/POSIX/BRE";
import * as ERE from "@mbse/expressions/POSIX/ERE";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const char = (c: string) => (u: any) => u.character((r: any) => r.value(BigInt(c.codePointAt(0)!)));
const alternative = (...elements: unknown[]) => (u: any) => u.alternative((r: any) => r.elements(elements));
const pattern = (...alternatives: unknown[]) => (u: any) => u.pattern((r: any) => r.alternatives(alternatives));
const anchor = (kind: string) => (u: any) => u.assertion((r: any) => r.assertionKind(kind));
const DIALECT = ERE.Expressions.DIALECT;

/** What `evaluate` says of `expression`: its value, or the reason it refuses it. */
function refused(evaluate: (expression: any) => unknown, expression: any): string {
  try {
    return String(evaluate(expression));
  } catch (error) {
    return (error as Error).message;
  }
}

<a id="T08.12.1"></a>
## 08.12.1 · An ERE, `ab|c` and `^(a|b)+c?$`

```text
ab|c        ^(a|b)+c?$
```

An ERE's kinds are a BRE's, named alike, but for its backreference, which it lacks: `ab|c` is a `pattern` of two
`alternative`s; `^(a|b)+c?$` holds anchors, a `capturinggroup` of two alternatives, and two `quantifier`s, `+` (at least
one) and `?` (at most one). `ERE.Text.ToText` writes each as an ERE does, its operators unescaped; the same tree, held by
a BRE's holder, prints as a BRE writes it:

In [2]:
const choice = pattern(alternative(char("a"), char("b")), alternative(char("c")));
const grouped = pattern(alternative(
  anchor("start"),
  (u: any) => u.quantifier((r: any) => r.min(1n).element((u: any) => u.capturinggroup(
    (g: any) => g.alternatives([alternative(char("a")), alternative(char("b"))])))),
  (u: any) => u.quantifier((r: any) => r.min(0n).max(1n).element(char("c"))),
  anchor("end")));
const [eres, bres]: [any[], any[]] = store.transaction(() => [
  [choice, grouped].map((spec) => S.POSIX.ERE.Expression((e: any) => e.root(spec))),
  [choice, grouped].map((spec) => S.POSIX.BRE.Expression((e: any) => e.root(spec))),
]);
eres.forEach((ere, i) => {
  console.log(ERE.Text.ToText(ere).padEnd(12), `as a BRE: ${BRE.Text.ToText(bres[i]).padEnd(26)}`,
    [...new Set([...Terms.walk(ere)].map((t) => Terms.kindOf(t)[0]))].join(", "));
});
console.log(DIALECT.name(), DIALECT.tag, "| its kinds:", [...DIALECT.kinds().keys()].join(", "));
assert.deepEqual(eres.map((x) => ERE.Text.ToText(x)), ["ab|c", "^(a|b)+c?$"]);

ab|c         as a BRE: ab\|c                      pattern, alternative, character


^(a|b)+c?$   as a BRE: ^\(a\|b\)\{1,\}c\{0,1\}$   pattern, alternative, assertion, quantifier, capturinggroup, character


POSIX ERE posix_ere | its kinds: patternlist, pattern, alternative, capturinggroup, quantifier, characterclass, characterclassrange, characterset, character, assertion, exclass, collatingelement, equivalence, error, missing


<a id="T08.12.2"></a>
## 08.12.2 · What an ERE has, and what it lacks

```text
a^$b        a+{0,1}        x*?y{1,3}?
```

An ERE's grammar lets an anchor stand anywhere, so `a^$b` is a valid ERE, which matches nothing (XBD 9.4.9); and it lets a
quantifier repeat a repeated expression, `a+{0,1}`, which the printer writes so that it reads back the same, never as
`a+?`. That would be a lazy quantifier, POSIX.1-2024's repetition modifier, which the dialect holds as a quantifier
that is not `greedy`. It has no backreference: the dialect's `Term` has no such kind, so the builders refuse to write
one into it:

In [3]:
const held: any[] = store.transaction(() => [
  [char("a"), anchor("start"), anchor("end"), char("b")],
  [(u: any) => u.quantifier((r: any) => r.min(0n).max(1n).element((u: any) => u.quantifier((q: any) => q.min(1n).element(char("a")))))],
  [(u: any) => u.quantifier((r: any) => r.min(0n).greedy(false).element(char("x"))),
    (u: any) => u.quantifier((r: any) => r.min(1n).max(3n).greedy(false).element(char("y")))],
].map((spec) => S.POSIX.ERE.Expression((e: any) => e.root(pattern(alternative(...spec))))));
for (const expression of held) {
  console.log(ERE.Text.ToText(expression).padEnd(10), "|", DIALECT.validate(expression).join("; ") || "valid");
}
try {
  store.transaction(() => S.POSIX.ERE.Expression((e: any) => e.root((u: any) => u.backreference((r: any) => r.ref(1n)))));
} catch {
  console.log("\\1 refused by the builders | a kind of POSIX ERE:", DIALECT.kinds().has("backreference") ? "yes" : "no");
}
assert.deepEqual(held.map((x) => ERE.Text.ToText(x)), ["a^$b", "a+{0,1}", "x*?y{1,3}?"]);

a^$b       | valid


a+{0,1}    | valid


x*?y{1,3}? | valid


\1 refused by the builders | a kind of POSIX ERE: no


<a id="T08.12.3"></a>
## 08.12.3 · egrep's lists, and awk's escapes

```text
grep -E -e '^a' -e 'ab|c'        awk '/\ta+/'
```

`egrep` and `grep -E` read a list of EREs, a line each, as grep reads BREs: the same kind, `patternlist`, of this
dialect. awk reads EREs too, with one difference in how they are written (XCU awk, "Regular Expressions"): a character
may be written as C writes it in a string, `\t`, `\n`, `\001`. That is a way of writing an ERE, not a grammar of its
own, so awk has no dialect: `ERE.Text.text`, given `awk`, prints a term with C's escapes, where an ERE writes a tab as
itself (`⇥` below):

In [4]:
const [listed, tabbed] = store.transaction(() => [
  S.POSIX.ERE.Expression((e: any) => e.root((u: any) => u.patternlist((r: any) => r.patterns([
    pattern(alternative(anchor("start"), char("a"))), choice])))),
  S.POSIX.ERE.Expression((e: any) => e.root(pattern(alternative(
    char("\t"), (u: any) => u.quantifier((r: any) => r.min(1n).element(char("a"))))))),
]);
console.log(ERE.Text.ToText(listed));
console.log("as an ERE:", ERE.Text.ToText(tabbed).replaceAll("\t", "⇥"), "| as awk writes it:", ERE.Text.text(tabbed, true));
assert.ok(ERE.Text.ToText(listed) === "^a\nab|c" && ERE.Text.text(tabbed, true) === "\\ta+");

^a
ab|c


as an ERE: ⇥a+ | as awk writes it: \ta+


<a id="T08.12.4"></a>
## 08.12.4 · Checked as its grammar has it

Validation checks what each kind's terms may be, as XBD 9.5.3's grammar has them, and its attributes' values: a
bracket expression holds no `.` as a set, since within brackets `.` is a character; an ERE has `.` alone among sets,
no `\w`; an equivalence class's name holds no `=`. An ERE binds nothing, as a BRE binds nothing:

In [5]:
const checked: Record<string, any> = store.transaction(() => Object.fromEntries(([
  ["[.]", (u: any) => u.characterclass((c: any) => c.elements([(u: any) => u.characterset((s: any) => s.setKind("any"))]))],
  ["\\w", (u: any) => u.characterset((s: any) => s.setKind("word"))],
  ["[[==e=]]", (u: any) => u.characterclass((c: any) => c.elements([(u: any) => u.equivalence((x: any) => x.name("=e"))]))],
  ["(a)*", (u: any) => u.quantifier((r: any) => r.min(0n).element(
    (u: any) => u.capturinggroup((g: any) => g.alternatives([alternative(char("a"))]))))],
] as [string, unknown][]).map(([source, spec]) => [source, S.POSIX.ERE.Expression((e: any) => e.root(pattern(alternative(spec))))])));
for (const [source, expression] of Object.entries(checked)) {
  console.log(source.padEnd(8), "|", DIALECT.validate(expression).join("; ") || "valid",
    "| needs:", [...Symbolics.free(expression, DIALECT)].sort().join(", ") || "nothing");
}
assert.deepEqual(DIALECT.validate(checked["(a)*"]), []);

[.]      | argument 0: argument 0: a characterclass's elements cannot hold a characterset | needs: nothing


\w       | argument 0: argument 0: CharacterSet.setKind cannot be 'word' | needs: nothing


[[==e=]] | argument 0: argument 0: argument 0: an equivalence's name cannot hold '=' | needs: nothing


(a)*     | valid | needs: nothing


<a id="T08.12.5"></a>
## 08.12.5 · Read by C++, and never evaluated

```text
std::regex r("^(a|b)+c?$", std::regex::extended);        std::regex r("^a\nab|c", std::regex::egrep);
```

A C++ `regex` whose flags name `extended`, `awk` or `egrep` holds a POSIX ERE term, which `Cpp RegExp` validates as
POSIX ERE does, then checks what C++ selects of it: a list only under `egrep`, and no lazy quantifier, the repetition
modifier the POSIX editions C++ cites lack. Nothing evaluates an ERE; the framework's interpreter refuses each kind by its
dialect's name:

In [6]:
const twoLines = (u: any) => u.patternlist((r: any) => r.patterns([pattern(alternative(anchor("start"), char("a"))), choice]));
const regexes: Record<string, any> = store.transaction(() => Object.fromEntries(([
  ["extended: ^(a|b)+c?$", "extended", grouped],
  ["egrep: ^a, ab|c", "egrep", twoLines],
  ["awk: ^a, ab|c", "awk", twoLines],
  ["extended: x*?", "extended", pattern(alternative(
    (u: any) => u.quantifier((r: any) => r.min(0n).greedy(false).element(char("x")))))],
] as [string, string, unknown][]).map(([source, flag, spec]) => [source, S.Cpp.RegExp.Expression((e: any) => e.root((u: any) => u.regex(
  (r: any) => r.flags([flag]).pattern((t: any) => t.posix_ere(spec)))))])));
for (const [source, regex] of Object.entries(regexes)) {
  console.log(source.padEnd(21), "|", CppRegExp.Expressions.DIALECT.validate(regex).join("; ") || "valid");
}
const interpreter = new Simplifiers.Interpreter(DIALECT, new Map(), { evaluated: new Set() }); // no kind evaluated
console.log(refused((x) => interpreter.run(x), Terms.term(eres[1])));
assert.deepEqual(CppRegExp.Expressions.DIALECT.validate(regexes["extended: ^(a|b)+c?$"]), []);

extended: ^(a|b)+c?$  | valid


egrep: ^a, ab|c       | valid


awk: ^a, ab|c         | a patternlist is not in the awk grammar


extended: x*?         | a lazy quantifier is not in the extended grammar


POSIX ERE pattern is not evaluated


<a id="T08.12.6"></a>
## 08.12.6 · Its editions

`POSIX ERE` covers the same editions as `POSIX BRE`, POSIX.1's since 2001, each by its designation and its common
names, the ISO/IEC designations C++ cites among them; each edition has every one of its kinds
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):

In [7]:
const ere = store.Instances.Expressions.POSIX.ERE; // the dialect, by its global name
for (const [designation, edition] of Standards.standards(ere)) {
  console.log(designation.padEnd(21), "|", Standards.tags(edition).slice(1).join(", "), "|", Standards.kinds(edition).length, "kinds");
}
assert.deepEqual([...Standards.standards(ere).keys()], [...Standards.standards(store.Instances.Expressions.POSIX.BRE).keys()]);
assert.ok(Standards.find(ere, "ISO/IEC/IEEE 9945:2009") === Standards.find(ere, "POSIX.1-2008"));

IEEE Std 1003.1-2001  | POSIX.1-2001, ISO/IEC 9945:2003 | 13 kinds


IEEE Std 1003.1-2008  | POSIX.1-2008, ISO/IEC/IEEE 9945:2009 | 13 kinds


IEEE Std 1003.1-2017  | POSIX.1-2017 | 13 kinds


IEEE Std 1003.1-2024  | POSIX.1-2024 | 13 kinds


## What you learned

- **An ERE is a `POSIX ERE` term**, its kinds a BRE's but for the backreference it lacks, printed with its operators
  unescaped.
- **Its grammar is its own**: an anchor anywhere, a repeated quantifier, POSIX.1-2024's lazy quantifier held.
- **egrep's list is a kind, awk's escapes a way of printing**: neither egrep nor awk is a dialect, since POSIX defines no
  grammar of theirs.
- **C++ reads an ERE** under its `extended`, `awk` and `egrep` grammars, checking what it selects of POSIX; nothing
  evaluates one.

That ends tutorial 8's sublanguages. Next, Part II puts expressions to work as constraints, starting with case study 9.

<!-- nav -->
[← 8.11 · POSIX BRE](08_11_POSIX_BRE.ipynb) · [Home](../../README.md) · [8.13 · SystemVerilog Types →](08_13_SystemVerilog_Types.ipynb)